In [1]:
import sys
sys.path.append('../')

from core import LADTransferTreeBoost, LSTransferTreeBoost, MTransferTreeBoost
import pandas as pd
from sklearn.model_selection import train_test_split
import xgboost as xgb
import numpy as np
from utils import * #only needed for xgboost
import itertools
import matplotlib.pyplot as plt
from baselines import *

In [2]:
test_size = 0.8 #0.8or 0.93
target_column = 'Dgv'
use_latvia = True

seed_list = [1]

In [3]:
predictor_columns = ['pzabovezmean', 'pzabove2', 'zq5', 'zq10',
    'zq15', 'zq20', 'zq25', 'zq30', 'zq35', 'zq40', 'zq45', 'zq50', 'zq55',
    'zq60', 'zq65', 'zq70', 'zq75', 'zq80', 'zq85', 'zq90', 'zq95',
    'zpcum1', 'zpcum2', 'zpcum3', 'zpcum4', 'zpcum5', 'zpcum6', 'zpcum7',
    'zpcum8', 'zpcum9'
    ]

#random_size = np.random.randint(57, 59)
#predictor_columns = np.random.choice(predictor_columns, size = random_size)
predictor_columns

['pzabovezmean',
 'pzabove2',
 'zq5',
 'zq10',
 'zq15',
 'zq20',
 'zq25',
 'zq30',
 'zq35',
 'zq40',
 'zq45',
 'zq50',
 'zq55',
 'zq60',
 'zq65',
 'zq70',
 'zq75',
 'zq80',
 'zq85',
 'zq90',
 'zq95',
 'zpcum1',
 'zpcum2',
 'zpcum3',
 'zpcum4',
 'zpcum5',
 'zpcum6',
 'zpcum7',
 'zpcum8',
 'zpcum9']

In [4]:
#ablation study for transfertreeboost Gaussian errors, with gaussian source domain errors
ablation_transfer_real = pd.DataFrame(columns = ['seed', 'method',
                                   'v', 'source_tree_size', 'target_tree_size', 'k', 'm_0', 'rmse', 'mae'])

v_list = [0.05, 0.1]
source_tree_size_list = [1,2]
target_tree_size_list = [1,2]
k_list = [0.01, 0.05, 0.1]
m_0_list = [0.5, 0.9]


# --- Step 2: Create full parameter grid ---
param_grid = list(itertools.product(
    v_list,
    source_tree_size_list,
    target_tree_size_list,
    k_list,
    m_0_list
))

# --- Step 3: Sample random combinations ---
#sampled_configs = random.sample(param_grid, n_samples)

for seed in seed_list:

    #data from Svedala
    data_sweden = pd.read_csv(r'../datasets/rs_sweden.csv', index_col=[0])
    data_sweden = data_sweden[0:1000]

    



    

    #evaluate and rain on latvia instead (keep naming for simplicity)
    data_latvia = pd.read_csv(r'../datasets/rs_lettland.csv', index_col=[0])
    data_latvia = data_latvia.rename(columns = {'H_AVERAGE': 'Hgv', 'D_AVERAGE': 'Dgv', 'VOLUME': 'Volume'})
    data_train, data_temp = train_test_split(data_latvia, test_size=0.8, random_state=seed)
    data_val, data_test = train_test_split(data_temp, test_size=0.5, random_state=seed)

    #"General" base dataset (to use for transfer)
    X_source_train = np.array(data_sweden[predictor_columns])
    y_source_train = np.array(data_sweden[target_column])

    #Specific train and test set
    X_target_train = np.array(data_train[predictor_columns])
    y_target_train = np.array(data_train[target_column])

    X_target_val = np.array(data_val[predictor_columns])
    y_target_val = np.array(data_val[target_column])

    X_target_test = np.array(data_test[predictor_columns])
    y_target_test = np.array(data_test[target_column])

    print(len(X_target_train), len(X_target_val), len(X_target_test))
    for config in param_grid:
        v, source_tree_size, target_tree_size, k, m_0 = config


        #Test for all methods!!!!

        method = f'LSTransferTreeBoost'
        fiter = LSTransferTreeBoost(epochs=1000, v=v, source_tree_size=source_tree_size, 
                                    target_tree_size=target_tree_size, k=k, m_0=m_0)
        fiter.fit(X_target_train, y_target_train, X_source_train, y_source_train, val_x=X_target_val, val_y=y_target_val, early_stopping_rounds=8, show_curves = False)
        rmse = fiter.evaluate(X_target_test, y_target_test, metric = 'rmse')
        mae = fiter.evaluate(X_target_test, y_target_test, metric = 'mae')

        ablation_transfer_real.loc[len(ablation_transfer_real)] = [seed, method, v, source_tree_size, target_tree_size, k, m_0, rmse, mae]
        ablation_transfer_real.to_csv(f'ablation_transfer_real.csv')


      
                        
    



C:\Users\Dag Bjornberg\AppData\Local\Temp\ipykernel_15648\3166160071.py:27: DtypeWarning: Columns (4,5) have mixed types. Specify dtype option on import or set low_memory=False.
  data_sweden = pd.read_csv(r'../datasets/rs_sweden.csv', index_col=[0])


380 760 760


KeyboardInterrupt: 

In [ ]:
#ablation study for xgboost Gaussian errors, with gaussian source domain errors
ablation_transfer_real = pd.DataFrame(columns = ['seed', 'method',
                                   'v', 'target_tree_size', 'rmse', 'mae'])

v_list = [0.005, 0.01, 0.02, 0.03, 0.05, 0.08, 0.1, 0.12, 0.15]
target_tree_size_list = [1,2,3,4,5,6,7]

# --- Step 2: Create full parameter grid ---
param_grid = list(itertools.product(
    v_list,
    target_tree_size_list))

# --- Step 3: Sample random combinations ---
#sampled_configs = random.sample(param_grid, n_samples)

for seed in seed_list:

        #data from Svedala
    data_sweden = pd.read_csv(r'../datasets/rs_sweden.csv', index_col=[0])

    



    

    #evaluate and rain on latvia instead (keep naming for simplicity)
    data_latvia = pd.read_csv(r'../datasets/rs_lettland.csv', index_col=[0])
    data_latvia = data_latvia.rename(columns = {'H_AVERAGE': 'Hgv', 'D_AVERAGE': 'Dgv', 'VOLUME': 'Volume'})
    data_train, data_temp = train_test_split(data_latvia, test_size=0.95, random_state=seed)
    data_val, data_test = train_test_split(data_temp, test_size=0.5, random_state=seed)

    #"General" base dataset (to use for transfer)
    X_source_train = np.array(data_sweden[predictor_columns])
    y_source_train = np.array(data_sweden[target_column])

    #Specific train and test set
    X_target_train = np.array(data_train[predictor_columns])
    y_target_train = np.array(data_train[target_column])

    X_target_val = np.array(data_val[predictor_columns])
    y_target_val = np.array(data_val[target_column])

    X_target_test = np.array(data_test[predictor_columns])
    y_target_test = np.array(data_test[target_column])
    print(len(X_target_train), len(X_target_val), len(X_target_test))
    for config in param_grid:
        v, target_tree_size = config



        #We also append baseline results!!
        method = 'xgboost'
        params = {
            'objective': 'reg:squarederror',  # Regression with squared error
            'max_depth': target_tree_size,                   # Maximum depth of a tree
            'eta': v,                       # Learning rate
            'eval_metric': 'rmse',           # RMSE as evaluation metric
            }
                
        bst = train_xgboost(X_target_train, y_target_train, X_target_val, y_target_val, boosting_rounds=1000, params=params)
        preds = test_xgboost(X_target_test, bst)
        rmse = compute_rmse(preds, y_target_test)
        mae = compute_mae(preds, y_target_test)
        ablation_transfer_real.loc[len(ablation_transfer_real)] = [seed, method, v, target_tree_size, rmse,
                            mae]
        
        ablation_transfer_real.to_csv(f'ablation_transfer_real_xgboost.csv')

        method = 'xgboost_naive_transfer'
        params = {
            'objective': 'reg:squarederror',  # Regression with squared error
            'max_depth': target_tree_size,                   # Maximum depth of a tree
            'eta': v,                       # Learning rate
            'eval_metric': 'rmse',           # RMSE as evaluation metric
            }
        X_comb = np.concatenate((X_target_train, X_source_train)) 
        y_comb = np.concatenate((y_target_train, y_source_train))       
        bst = train_xgboost(X_comb, y_comb, X_target_val, y_target_val, boosting_rounds=1000, params=params)
        preds = test_xgboost(X_target_test, bst)
        rmse = compute_rmse(preds, y_target_test)
        mae = compute_mae(preds, y_target_test)
        ablation_transfer_real.loc[len(ablation_transfer_real)] = [seed, method, v, target_tree_size, rmse,
                            mae]
        
        ablation_transfer_real.to_csv(f'ablation_transfer_real_xgboost.csv')

C:\Users\Dag Bjornberg\AppData\Local\Temp\ipykernel_5300\3003397196.py:19: DtypeWarning: Columns (4,5) have mixed types. Specify dtype option on import or set low_memory=False.
  data_sweden = pd.read_csv(r'../datasets/rs_sweden.csv', index_col=[0])


95 902 903


In [ ]:
#Find best hyperparameter values
dat_abl = pd.read_csv(f'ablation_transfer_real_xgboost.csv')
dat_abl_xgboost = dat_abl[dat_abl['method']=='xgboost']
dat_abl_naive = dat_abl[dat_abl['method']=='xgboost_naive_transfer']
dat_abl_xgboost = dat_abl_xgboost.sort_values(by = ['target_tree_size', 'v']).reset_index(drop=True)
group_index = dat_abl_xgboost.index // len(seed_list)

# Group by the group index and sum, then broadcast to original rows
dat_abl_xgboost['total_rmse'] = dat_abl_xgboost.groupby(group_index)['rmse'].transform('sum') / len(seed_list) #or how many seeds we have used!
dat_abl_xgboost['total_mae'] = dat_abl_xgboost.groupby(group_index)['mae'].transform('sum') / len(seed_list) #or how many seeds we have used!
dat_abl_xgboost= dat_abl_xgboost.sort_values(by = ['total_rmse']) #select on total_mae or total_rmse
#now, fin best parameters for each disturbance_level (we choose top 3)
optimal_params_transfer = pd.DataFrame(columns = ['target_tree_size', 'v', 'total_rmse', 'total_mae'])
best_params = dat_abl_xgboost[['target_tree_size', 'v', 'total_rmse', 'total_mae']].iloc[[0, len(seed_list), len(seed_list)*2]]
optimal_params_transfer = pd.concat((optimal_params_transfer, best_params))

optimal_params_transfer['total_rmse'] = np.round(optimal_params_transfer['total_rmse'], 2)
optimal_params_transfer['total_mae'] = np.round(optimal_params_transfer['total_mae'], 2)
optimal_params_transfer

C:\Users\Dag Bjornberg\AppData\Local\Temp\ipykernel_5300\4260450512.py:15: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  optimal_params_transfer = pd.concat((optimal_params_transfer, best_params))


,target_tree_size,v,total_rmse,total_mae
8,1,0.15,7.58,4.84
6,1,0.10,7.60,4.85
5,1,0.08,7.61,4.87


In [ ]:
dat_abl_naive = dat_abl_naive.sort_values(by = ['target_tree_size', 'v']).reset_index(drop=True)
group_index = dat_abl_naive.index // len(seed_list)

# Group by the group index and sum, then broadcast to original rows
dat_abl_naive['total_rmse'] = dat_abl_naive.groupby(group_index)['rmse'].transform('sum') / len(seed_list) #or how many seeds we have used!
dat_abl_naive['total_mae'] = dat_abl_naive.groupby(group_index)['mae'].transform('sum') / len(seed_list) #or how many seeds we have used!
dat_abl_naive= dat_abl_naive.sort_values(by = ['total_rmse']) #select on total_mae or total_rmse
#now, fin best parameters for each disturbance_level (we choose top 3)
optimal_params_transfer = pd.DataFrame(columns = ['target_tree_size', 'v', 'total_rmse', 'total_mae'])
best_params = dat_abl_naive[['target_tree_size', 'v', 'total_rmse', 'total_mae']].iloc[[0, len(seed_list), len(seed_list)*2]]
optimal_params_transfer = pd.concat((optimal_params_transfer, best_params))

optimal_params_transfer['total_rmse'] = np.round(optimal_params_transfer['total_rmse'], 2)
optimal_params_transfer['total_mae'] = np.round(optimal_params_transfer['total_mae'], 2)
optimal_params_transfer

C:\Users\Dag Bjornberg\AppData\Local\Temp\ipykernel_5300\3297914626.py:11: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  optimal_params_transfer = pd.concat((optimal_params_transfer, best_params))


,target_tree_size,v,total_rmse,total_mae
55,7,0.010,7.63,5.42
54,7,0.005,7.63,5.43
56,7,0.020,7.64,5.45


In [ ]:
#Find best hyperparameter values
dat_abl = pd.read_csv(f'ablation_transfer_real.csv')
dat_abl = dat_abl.sort_values(by = ['v', 'source_tree_size', 'target_tree_size', 'k', 'm_0']).reset_index(drop=True)
group_index = dat_abl.index // len(seed_list)

# Group by the group index and sum, then broadcast to original rows
dat_abl['total_rmse'] = dat_abl.groupby(group_index)['rmse'].transform('sum') / len(seed_list) #or how many seeds we have used!
dat_abl['total_mae'] = dat_abl.groupby(group_index)['mae'].transform('sum') / len(seed_list) #or how many seeds we have used!
dat_abl= dat_abl.sort_values(by = ['total_rmse']) #select on total_mae or total_rmse
#now, fin best parameters for each disturbance_level (we choose top 3)
optimal_params_transfer = pd.DataFrame(columns = ['v', 'source_tree_size', 'target_tree_size', 'k', 'm_0', 'total_rmse', 'total_mae'])
best_params = dat_abl[['v', 'source_tree_size', 'target_tree_size', 'k', 'm_0', 'total_rmse', 'total_mae']].iloc[[0, len(seed_list), len(seed_list)*2]]
optimal_params_transfer = pd.concat((optimal_params_transfer, best_params))

optimal_params_transfer['total_rmse'] = np.round(optimal_params_transfer['total_rmse'], 2)
optimal_params_transfer['total_mae'] = np.round(optimal_params_transfer['total_mae'], 2)
optimal_params_transfer

C:\Users\Dag Bjornberg\AppData\Local\Temp\ipykernel_5300\1491343038.py:13: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  optimal_params_transfer = pd.concat((optimal_params_transfer, best_params))


,v,source_tree_size,target_tree_size,k,m_0,total_rmse,total_mae
25,0.10,2,1,0.01,0.9,7.31,4.61
17,0.10,1,1,0.01,0.9,7.34,4.64
1,0.05,1,1,0.01,0.9,7.35,4.65
